# Comparación de Entrenamiento de Redes Neuronales: Basic vs Arnovi vs Diego

Este cuaderno entrena tres arquitecturas de redes neuronales diferentes en el dataset MNIST con hiperparámetros idénticos y visualiza su desempeño usando Altair.

## Descripción General de las Redes:
- **BasicNeuralNetwork**: Red neuronal simple con un único paso de entrenamiento
- **ArnoviNeuralNetwork**: Aprendizaje distribuido con promediado de pesos post-entrenamiento
- **DiegoNeuralNetwork**: Aprendizaje federado con sincronización por época

## 1. Importar Librerías Requeridas

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import altair as alt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Habilitar Altair para manejar datasets grandes
alt.data_transformers.disable_max_rows()

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Cargar Módulos de Redes Neuronales y Utilidades del Dataset

In [2]:
# Añadir el directorio padre al path para acceder al paquete Utils
notebook_dir = os.path.dirname(os.path.abspath('../__file__'))
sys.path.insert(0, os.path.abspath(notebook_dir))

# Importar funciones del dataset y utilidades desde el paquete Utils
from Utils.DatasetHandling import cargar_mnist, preprocesar
from Utils.Fuctions import forward, backward, cross_entropy, precision, predecir
from Utils.WeightsHandling import inicializar_pesos, actualizar_pesos

# Importar funciones de entrenamiento desde el paquete Neural Network (importar como módulo)
nn_path = os.path.join(notebook_dir, 'Neural Network')
if nn_path not in sys.path:
    sys.path.insert(0, nn_path)

from BasicNeuralNetwork import entrenar as entrenar_basic
from ArnoviNeuralNetwork import entrenar_arnovi
from DiegoNeuralNetwork import entrenar_diego

print("✓ Todos los módulos importados correctamente")

✓ Todos los módulos importados correctamente


## 3. Definir Parámetros de Entrenamiento Compartidos

Las tres redes se entrenarán con hiperparámetros idénticos para garantizar una comparación justa.

In [3]:
# ========================================
# HIPERPARÁMETROS COMPARTIDOS PARA TODAS LAS REDES
# ========================================

LEARNING_RATE = 0.5          # Tasa de aprendizaje para descenso de gradiente
NUM_PARTICIONES = 2          # Número de particiones (para Arnovi y Diego)
EPOCAS = 50                  # Número de épocas de entrenamiento
INTERVALO_LOG = 10           # Intervalo de registro para mostrar progreso

# Mostrar configuración
print("="*60)
print("  CONFIGURACIÓN DE ENTRENAMIENTO COMPARTIDA")
print("="*60)
print(f"  Tasa de Aprendizaje      : {LEARNING_RATE}")
print(f"  Número de Épocas         : {EPOCAS}")
print(f"  Particiones (Arnovi/Diego): {NUM_PARTICIONES}")
print(f"  Intervalo de Log         : {INTERVALO_LOG}")
print("="*60)

  CONFIGURACIÓN DE ENTRENAMIENTO COMPARTIDA
  Tasa de Aprendizaje      : 0.5
  Número de Épocas         : 50
  Particiones (Arnovi/Diego): 2
  Intervalo de Log         : 10


## 4. Cargar y Preparar Dataset MNIST

In [4]:
print("\n" + "="*60)
print("  CARGANDO DATASET MNIST")
print("="*60)

# Cargar MNIST
X_all, y_all = cargar_mnist('../data')

# Preprocesar: aplanar, normalizar, codificación one-hot y dividir 70/30
print("\nPreparando dataset...")
X_train, Y_train, y_train, X_test, Y_test, y_test = preprocesar(X_all, y_all)

print("\n✓ Dataset cargado y preparado correctamente")


  CARGANDO DATASET MNIST
Descargando MNIST con torchvision...
  Dataset completo: 70000 imágenes de 28×28 píxeles

Preparando dataset...

  Muestras de entrenamiento : 49000
  Muestras de prueba        : 21000
  Forma de X_train          : (49000, 784)  ← (muestras, píxeles)
  Forma de Y_train          : (49000, 10)  ← (muestras, clases)

✓ Dataset cargado y preparado correctamente


## 5. Funciones Auxiliares para Capturar Historial de Entrenamiento

Crearemos funciones contenedoras que capturen el historial de entrenamiento para cada red.

In [5]:
def train_basic_network(X_train, Y_train, y_train, X_test, Y_test, y_test,
                        epocas, lr, intervalo_log, model_name='BasicNN'):
    """
    Entrenar Red Neuronal Básica y capturar historial de entrenamiento.
    """
    print("\n" + "="*70)
    print(f"  ENTRENANDO: {model_name}")
    print("="*70)
    
    W1, b1, W2, b2 = inicializar_pesos()
    historial_loss = []
    historial_acc = []
    historial_acc_test = []
    
    for epoca in range(1, epocas + 1):
        # Paso hacia adelante
        Z1, A1, Z2, A2 = forward(X_train, W1, b1, W2, b2)
        
        # Calcular pérdida
        loss = cross_entropy(A2, Y_train)
        historial_loss.append(loss)
        
        # Calcular precisión
        y_pred_train = np.argmax(A2, axis=1)
        acc_train = precision(y_pred_train, y_train)
        historial_acc.append(acc_train)
        
        # Paso hacia atrás
        dW1, db1, dW2, db2 = backward(X_train, Y_train, Z1, A1, A2, W2)
        
        # Actualizar pesos
        W1, b1, W2, b2 = actualizar_pesos(W1, b1, W2, b2, dW1, db1, dW2, db2, lr)
        
        # Precisión en test
        y_pred_test = predecir(X_test, W1, b1, W2, b2)
        acc_test = precision(y_pred_test, y_test)
        historial_acc_test.append(acc_test)
        
        # Log de progreso
        if epoca % intervalo_log == 0 or epoca == 1:
            print(f"  Época {epoca:4d}/{epocas} │ "
                  f"Loss: {loss:.4f} │ "
                  f"Acc Train: {acc_train:.1f}% │ "
                  f"Acc Test: {acc_test:.1f}%")
    
    print(f"\n✓ Entrenamiento de {model_name} completado")
    
    return {
        'name': model_name,
        'loss': historial_loss,
        'accuracy': historial_acc,
        'accuracy_test': historial_acc_test
    }


def train_arnovi_network(X_train, Y_train, y_train, X_test, Y_test, y_test,
                         num_particiones, epocas, lr, intervalo_log, model_name='ArnoviNN'):
    """
    Entrenar Red Neuronal de Arnovi y capturar historial de entrenamiento.
    """
    print("\n" + "="*70)
    print(f"  ENTRENANDO: {model_name}")
    print("="*70)
    
    from ArnoviNeuralNetwork import particionar_dataset, entrenar_particion, promediar_pesos
    
    # Inicializar pesos
    W1_init, b1_init, W2_init, b2_init = inicializar_pesos()
    
    # Particionar dataset
    particiones = particionar_dataset(X_train, Y_train, y_train, num_particiones)
    
    # Entrenar cada partición
    lista_pesos_entrenados = []
    historiales_loss = []
    historiales_acc = []
    
    for k, (X_k, Y_k, y_k) in enumerate(particiones):
        W1_k, b1_k, W2_k, b2_k, hist_loss_k, hist_acc_k = entrenar_particion(
            X_k, Y_k, y_k,
            W1_init, b1_init, W2_init, b2_init,
            id_particion=k + 1,
            epocas=epocas,
            lr=lr,
            intervalo_log=intervalo_log,
            X_test=X_test,
            y_test=y_test
        )
        lista_pesos_entrenados.append((W1_k, b1_k, W2_k, b2_k))
        historiales_loss.append(hist_loss_k)
        historiales_acc.append(hist_acc_k)
    
    # Promediar pesos
    W1_final, b1_final, W2_final, b2_final = promediar_pesos(lista_pesos_entrenados)
    
    # Calcular precisión en test para el modelo promediado
    historial_acc_test = []
    for epoca in range(epocas):
        # Para cada época, calculamos la precisión promediando las precisiones individuales de las particiones
        # y también calculamos la precisión del modelo final
        y_pred_test = predecir(X_test, W1_final, b1_final, W2_final, b2_final)
        acc_test = precision(y_pred_test, y_test)
        historial_acc_test.append(acc_test)
    
    # Promediar pérdida y precisión entre particiones
    historial_loss_promedio = np.mean(np.array(historiales_loss), axis=0).tolist()
    historial_acc_promedio = np.mean(np.array(historiales_acc), axis=0).tolist()
    
    print(f"\n✓ Entrenamiento de {model_name} completado")
    
    return {
        'name': model_name,
        'loss': historial_loss_promedio,
        'accuracy': historial_acc_promedio,
        'accuracy_test': historial_acc_test
    }


def train_diego_network(X_train, Y_train, y_train, X_test, Y_test, y_test,
                        num_particiones, epocas, lr, intervalo_log, model_name='DiegoNN'):
    """
    Entrenar Red Neuronal de Diego y capturar historial de entrenamiento.
    """
    print("\n" + "="*70)
    print(f"  ENTRENANDO: {model_name}")
    print("="*70)
    
    from DiegoNeuralNetwork import particionar_dataset, train_on_batch, promediar_pesos
    
    # Inicializar pesos
    W1, b1, W2, b2 = inicializar_pesos()
    
    # Particionar dataset
    particiones = particionar_dataset(X_train, Y_train, y_train, num_particiones)
    
    # Bucle de entrenamiento
    historial_loss = []
    historial_acc = []
    historial_acc_test = []
    
    for epoca in range(1, epocas + 1):
        # Entrenar cada partición y recopilar pesos
        lista_pesos_epoca = []
        
        for k, (X_k, Y_k, y_k) in enumerate(particiones):
            W1_k, b1_k, W2_k, b2_k = train_on_batch(X_k, Y_k, W1, b1, W2, b2, lr)
            lista_pesos_epoca.append((W1_k, b1_k, W2_k, b2_k))
        
        # Promediar pesos
        W1, b1, W2, b2 = promediar_pesos(lista_pesos_epoca)
        
        # Calcular métricas en todo el conjunto de entrenamiento
        Z1_all, A1_all, Z2_all, A2_all = forward(X_train, W1, b1, W2, b2)
        loss = cross_entropy(A2_all, Y_train)
        acc_train = precision(np.argmax(A2_all, axis=1), y_train)
        
        historial_loss.append(loss)
        historial_acc.append(acc_train)
        
        # Precisión en test
        y_pred_test = predecir(X_test, W1, b1, W2, b2)
        acc_test = precision(y_pred_test, y_test)
        historial_acc_test.append(acc_test)
        
        # Log de progreso
        if epoca % intervalo_log == 0 or epoca == 1:
            print(f"  Época {epoca:4d}/{epocas} │ "
                  f"Loss: {loss:.4f} │ "
                  f"Acc Train: {acc_train:.1f}% │ "
                  f"Acc Test: {acc_test:.1f}%")
    
    print(f"\n✓ Entrenamiento de {model_name} completado")
    
    return {
        'name': model_name,
        'loss': historial_loss,
        'accuracy': historial_acc,
        'accuracy_test': historial_acc_test
    }

print("✓ Funciones de entrenamiento definidas")

✓ Funciones de entrenamiento definidas


## 6. Entrenar las Tres Redes con Parámetros Compartidos

In [6]:
# Iniciar cronómetro
start_time = datetime.now()

# Diccionario para almacenar resultados de todas las redes
training_results = {}

# 1. Entrenar Red Neuronal Básica
print("\n" + "█"*70)
print("  RED 1: RED NEURONAL BÁSICA")
print("█"*70)
training_results['BasicNN'] = train_basic_network(
    X_train, Y_train, y_train, X_test, Y_test, y_test,
    epocas=EPOCAS,
    lr=LEARNING_RATE,
    intervalo_log=INTERVALO_LOG
)

# 2. Entrenar Red Neuronal de Arnovi
print("\n" + "█"*70)
print("  RED 2: RED NEURONAL DE ARNOVI (Distribida con Promediado de Pesos)")
print("█"*70)
training_results['ArnoviNN'] = train_arnovi_network(
    X_train, Y_train, y_train, X_test, Y_test, y_test,
    num_particiones=NUM_PARTICIONES,
    epocas=EPOCAS,
    lr=LEARNING_RATE,
    intervalo_log=INTERVALO_LOG
)

# 3. Entrenar Red Neuronal de Diego
print("\n" + "█"*70)
print("  RED 3: RED NEURONAL DE DIEGO (Federada con Promediado por Época)")
print("█"*70)
training_results['DiegoNN'] = train_diego_network(
    X_train, Y_train, y_train, X_test, Y_test, y_test,
    num_particiones=NUM_PARTICIONES,
    epocas=EPOCAS,
    lr=LEARNING_RATE,
    intervalo_log=INTERVALO_LOG
)

# Calcular tiempo transcurrido
elapsed_time = datetime.now() - start_time

print("\n" + "="*70)
print("  ENTRENAMIENTO COMPLETADO")
print("="*70)
print(f"  Tiempo total: {elapsed_time}")
print("="*70)


██████████████████████████████████████████████████████████████████████
  RED 1: RED NEURONAL BÁSICA
██████████████████████████████████████████████████████████████████████

  ENTRENANDO: BasicNN

  W1: (784, 128)  (pesos entrada → oculta)
  b1: (1, 128)  (sesgos capa oculta)
  W2: (128, 10)  (pesos oculta → salida)
  b2: (1, 10)  (sesgos capa salida)
  Época    1/50 │ Loss: 2.4114 │ Acc Train: 12.4% │ Acc Test: 28.5%
  Época   10/50 │ Loss: 1.0095 │ Acc Train: 66.0% │ Acc Test: 69.6%
  Época   20/50 │ Loss: 0.5765 │ Acc Train: 82.2% │ Acc Test: 84.2%
  Época   30/50 │ Loss: 0.4543 │ Acc Train: 87.4% │ Acc Test: 87.0%
  Época   40/50 │ Loss: 0.3944 │ Acc Train: 88.7% │ Acc Test: 87.8%
  Época   50/50 │ Loss: 0.3537 │ Acc Train: 89.8% │ Acc Test: 89.6%

✓ Entrenamiento de BasicNN completado

██████████████████████████████████████████████████████████████████████
  RED 2: RED NEURONAL DE ARNOVI (Distribida con Promediado de Pesos)
███████████████████████████████████████████████████████████

## 7. Recopilar y Organizar Métricas de Entrenamiento

In [7]:
# Preparar datos para visualización
data_loss = []
data_accuracy = []
data_accuracy_test = []

for network_name, results in training_results.items():
    loss_values = results['loss']
    accuracy_values = results['accuracy']
    accuracy_test_values = results['accuracy_test']
    
    for epoch, (loss, acc, acc_test) in enumerate(zip(loss_values, accuracy_values, accuracy_test_values), 1):
        data_loss.append({
            'Epoch': epoch,
            'Network': network_name,
            'Loss': loss
        })
        
        data_accuracy.append({
            'Epoch': epoch,
            'Network': network_name,
            'Accuracy (%)': acc
        })
        
        data_accuracy_test.append({
            'Epoch': epoch,
            'Network': network_name,
            'Test Accuracy (%)': acc_test
        })

# Crear DataFrames
df_loss = pd.DataFrame(data_loss)
df_accuracy = pd.DataFrame(data_accuracy)
df_accuracy_test = pd.DataFrame(data_accuracy_test)

print("✓ Datos preparados para visualización")
print(f"\nFormas de los DataFrames:")
print(f"  Pérdida: {df_loss.shape}")
print(f"  Precisión de Entrenamiento: {df_accuracy.shape}")
print(f"  Precisión de Test: {df_accuracy_test.shape}")

✓ Datos preparados para visualización

Formas de los DataFrames:
  Pérdida: (150, 3)
  Precisión de Entrenamiento: (150, 3)
  Precisión de Test: (150, 3)


## 8. Visualizaciones Interactivas con Altair

In [8]:
# Gráfico de comparación de pérdida
chart_loss = alt.Chart(df_loss).mark_line(point=True, size=2.5).encode(
    x=alt.X('Epoch:Q', title='Época', scale=alt.Scale(nice=False)),
    y=alt.Y('Loss:Q', title='Pérdida de Entropía Cruzada'),
    color=alt.Color('Network:N', title='Red',
                    scale=alt.Scale(domain=['BasicNN', 'ArnoviNN', 'DiegoNN'],
                                   range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    strokeDash=alt.StrokeDash('Network:N', 
                             scale=alt.Scale(domain=['BasicNN', 'ArnoviNN', 'DiegoNN'],
                                           range=[[], [5, 5], [2, 2]])),
    tooltip=['Epoch:Q', 'Network:N', alt.Tooltip('Loss:Q', format='.4f')]
).properties(
    width=800,
    height=400,
    title='Comparación de Pérdida de Entrenamiento'
).interactive()

chart_loss.show()
print("✓ Gráfico de comparación de pérdida mostrado")

alt.Chart(...)

✓ Gráfico de comparación de pérdida mostrado


## 9. Comparación de Precisión de Entrenamiento

In [9]:
# Gráfico de comparación de precisión de entrenamiento
chart_acc = alt.Chart(df_accuracy).mark_line(point=True, size=2.5).encode(
    x=alt.X('Epoch:Q', title='Época', scale=alt.Scale(nice=False)),
    y=alt.Y('Accuracy (%):Q', title='Precisión de Entrenamiento (%)', 
            scale=alt.Scale(domain=[df_accuracy['Accuracy (%)'].min() - 5, 
                                   df_accuracy['Accuracy (%)'].max() + 5])),
    color=alt.Color('Network:N', title='Red',
                    scale=alt.Scale(domain=['BasicNN', 'ArnoviNN', 'DiegoNN'],
                                   range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    strokeDash=alt.StrokeDash('Network:N',
                             scale=alt.Scale(domain=['BasicNN', 'ArnoviNN', 'DiegoNN'],
                                           range=[[], [5, 5], [2, 2]])),
    tooltip=['Epoch:Q', 'Network:N', alt.Tooltip('Accuracy (%):Q', format='.2f')]
).properties(
    width=800,
    height=400,
    title='Comparación de Precisión de Entrenamiento'
).interactive()

chart_acc.show()
print("✓ Gráfico de comparación de precisión de entrenamiento mostrado")

alt.Chart(...)

✓ Gráfico de comparación de precisión de entrenamiento mostrado


## 10. Panel de Control de Visualización Combinada

In [10]:
# Crear un panel de control combinado con los tres gráficos
dashboard = alt.vconcat(
    chart_loss,
    chart_acc
).properties(
    title='Panel de Comparación de Redes Neuronales: BasicNN vs ArnoviNN vs DiegoNN'
).resolve_scale(
    color='shared'
)

# Mostrar el panel de control
dashboard.show()


alt.VConcatChart(...)